# Daily Modeling dataset EDA

Inspect the assembled dataset, rainfall distribution, temporal coverage, and station coverage. Set `FREQ` before running all cells.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FREQ = "weekly"
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Daily_Modeling").exists())
DATASET_PATH = PROJECT_DIR / "Daily_Modeling" / "output" / FREQ / "assembled" / f"{FREQ}_dataset_station_centered.npz"
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_PATH} does not exist. Run "
        f"`python -m Daily_Modeling.scripts.02_assemble_dataset --freq {FREQ}` first."
    )
DATASET_PATH

In [ ]:
with np.load(DATASET_PATH, allow_pickle=True) as raw:
    data = {key: raw[key] for key in raw.files}

summary = pd.DataFrame({
    "shape": {key: str(value.shape) for key, value in data.items()},
    "dtype": {key: str(value.dtype) for key, value in data.items()},
})
summary

In [ ]:
dates = pd.to_datetime({"year": data["years"], "month": data["months"], "day": data["days"]})
samples = pd.DataFrame({
    "date": dates,
    "station": data["stations"].astype(str),
    "rainfall_mm": data["rainfall_mm_raw"].reshape(-1),
})
samples.describe(include="all", datetime_is_numeric=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(samples["rainfall_mm"], bins=60)
axes[0].set(xlabel="Rainfall (mm)", ylabel="Samples", title=f"{FREQ.title()} rainfall")
axes[1].hist(np.log1p(samples["rainfall_mm"]), bins=60)
axes[1].set(xlabel="log1p rainfall", ylabel="Samples", title="Log-scale distribution")
fig.tight_layout()

In [ ]:
station_summary = samples.groupby("station").agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    samples=("rainfall_mm", "size"),
    mean_rainfall_mm=("rainfall_mm", "mean"),
    max_rainfall_mm=("rainfall_mm", "max"),
).sort_values("samples", ascending=False)
station_summary

In [ ]:
coverage = samples.assign(year=samples["date"].dt.year).pivot_table(
    index="station", columns="year", values="rainfall_mm", aggfunc="count", fill_value=0
)
fig, ax = plt.subplots(figsize=(14, max(5, 0.3 * len(coverage))))
image = ax.imshow(coverage > 0, aspect="auto", interpolation="nearest", cmap="Blues")
ax.set(yticks=range(len(coverage.index)), yticklabels=coverage.index, xlabel="Year index", ylabel="Station", title="Station-year coverage")
fig.tight_layout()